In [1]:
# ============================================================
# DAY 6 — WEEKLY REPORT WITH VOLUME, SECTOR & BEARISH SIGNALS
# AI Stock Screener — Indian Markets
# Input : data/price_data.pkl, data/indicator_data.pkl
#         data/fundamental_scores.csv, data/technical_report.csv
#         data/consolidation_info.csv
# Output: data/technical_report.csv (updated)
# ============================================================

# ── CELL 1: IMPORTS + LOAD DATA ──────────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

with open('../data/price_data.pkl', 'rb') as f:
    price_data = pickle.load(f)

with open('../data/indicator_data.pkl', 'rb') as f:
    indicator_data = pickle.load(f)

fund_df       = pd.read_csv('../data/fundamental_scores.csv')
tech_df       = pd.read_csv('../data/technical_report.csv')
consol_info_df = pd.read_csv('../data/consolidation_info.csv')

print(f"Price data     : {len(price_data)} stocks")
print(f"Indicator data : {len(indicator_data)} stocks")
print(f"Fundamental    : {len(fund_df)} stocks")
print(f"Tech report    : {len(tech_df)} stocks")

Price data     : 37 stocks
Indicator data : 37 stocks
Fundamental    : 37 stocks
Tech report    : 37 stocks


In [2]:
# ── CELL 2: VOLUME INFERENCE ──────────────────────────────────
def get_volume_inference(row):
    """
    Returns (vol_label, inference) based on setup type and volume ratio
    Uses Vol 5D Ratio (5-day avg vs prior 15-day avg)
    """
    vol   = float(row.get('Vol 5D Ratio', row.get('Vol Ratio', 1.0)))
    setup = str(row['Best Setup'])
    adx   = float(row['ADX'])

    if   vol >= 3.0: vol_label = 'Extremely High'
    elif vol >= 2.0: vol_label = 'Very High'
    elif vol >= 1.5: vol_label = 'High'
    elif vol >= 1.0: vol_label = 'Normal'
    elif vol >= 0.7: vol_label = 'Low'
    else:            vol_label = 'Very Low'

    # ── Momentum ──────────────────────────────────────────────
    if setup == 'Momentum':
        if   vol >= 2.0: return vol_label, 'Strong participation → trend likely to continue'
        elif vol >= 1.5: return vol_label, 'Good volume support → momentum confirmed'
        elif vol >= 1.0: return vol_label, 'Average volume → monitor for increase'
        elif vol >= 0.7: return vol_label, 'Weak volume → momentum may fade, wait'
        else:            return vol_label, 'Very low volume → no conviction, caution'

    # ── Reversal ──────────────────────────────────────────────
    elif setup == 'Reversal':
        if   vol <= 0.5: return vol_label, 'Sellers exhausted → reversal likely'
        elif vol <= 0.7: return vol_label, 'Low volume on down move → selling pressure easing'
        elif vol <= 1.0: return vol_label, 'Neutral volume → wait for low-volume base to form'
        elif vol <= 1.5: return vol_label, 'Elevated volume on reversal → watch direction'
        else:            return vol_label, 'High volume → could be capitulation or distribution'

    # ── Watching / Waiting ────────────────────────────────────
    else:
        if   adx > 40 and vol >= 2.0: return vol_label, 'Above avg volume but no setup → watch closely'
        elif adx > 40 and vol >= 1.5: return vol_label, 'Strong downtrend → no entry signal'
        elif adx > 40:                return vol_label, 'Strong downtrend → no entry signal'
        elif vol <= 0.5:              return vol_label, 'Very low volume → no interest, wait'
        elif vol >= 1.5 and adx < 35: return vol_label, 'Above avg volume but no setup → watch'
        elif vol >= 1.0:              return vol_label, 'Normal activity → no setup forming yet'
        elif vol >= 0.7:              return vol_label, 'Low activity → no setup forming yet'
        else:                         return vol_label, 'Very low volume → no interest, wait'


In [3]:
# ── CELL 3: CALCULATE 5-DAY VOLUME RATIO ─────────────────────
# Week Vol = last 5 days avg vs prior 15-day avg (non-overlapping)
vol_5d_data = {}
for symbol, df in price_data.items():
    try:
        if len(df) < 20:
            vol_5d_data[symbol] = 1.0
            continue
        vol_5d_avg      = df['Volume'].iloc[-5:].mean()
        vol_prior15_avg = df['Volume'].iloc[-20:-5].mean()
        vol_5d_ratio    = vol_5d_avg / vol_prior15_avg if vol_prior15_avg > 0 else 1.0
        vol_5d_data[symbol] = round(vol_5d_ratio, 2)
    except:
        vol_5d_data[symbol] = 1.0

tech_df['Vol 5D Ratio'] = tech_df['Symbol'].map(vol_5d_data)
print(f"Vol 5D Ratio calculated for {len(vol_5d_data)} stocks ✅")

Vol 5D Ratio calculated for 37 stocks ✅


In [4]:
# ── CELL 4: BREAKOUT VOLUME (consolidating stocks only) ───────
# Break Vol = this week avg vs consolidation period avg
breakout_vol_data = {}
for _, crow in consol_info_df.iterrows():
    symbol      = crow['Symbol']
    consol_days = int(crow['consol_days'])

    if symbol not in price_data:
        continue

    df = price_data[symbol].copy()

    if len(df) < consol_days + 5:
        breakout_vol_data[symbol] = 1.0
        continue

    week_vol   = df['Volume'].iloc[-5:].mean()
    consol_avg = df['Volume'].iloc[-consol_days:-5].mean()
    ratio      = week_vol / consol_avg if consol_avg > 0 else 1.0
    breakout_vol_data[symbol] = round(ratio, 2)

tech_df['Breakout Vol'] = tech_df['Symbol'].map(breakout_vol_data).fillna(0.0)
print(f"Breakout Vol calculated for {len(breakout_vol_data)} consolidating stocks ✅")



Breakout Vol calculated for 10 consolidating stocks ✅


In [5]:
# ── CELL 5: SECTOR INDEX MAP + FETCH ─────────────────────────
SECTOR_INDEX_MAP = {
    'Information Technology'            : '^CNXIT',
    'Healthcare'                        : '^CNXPHARMA',
    'Financial Services'                : 'FINIETF.NS',
    'Capital Goods'                     : '^CNXINFRA',
    'Chemicals'                         : 'MID150BEES.NS',   # CHEMIETF delisted
    'Consumer Durables'                 : '^CNXCONSUM',
    'Oil, Gas & Consumable Fuels'       : '^CNXENERGY',
    'Automobile and Auto Components'    : '^CNXAUTO',
    'Textiles'                          : 'MID150BEES.NS',
    'Consumer Services'                 : 'MID150BEES.NS',
    'Banking'                           : '^NSEBANK',
    'Private Banking'                   : 'PVTBANIETF.NS',
    'PSU Banking'                       : '^CNXPSUBANK',
    'Fast Moving Consumer Goods'        : '^CNXFMCG',
    'Metals & Mining'                   : '^CNXMETAL',
    'Realty'                            : '^CNXREALTY',
    'Media Entertainment & Publication' : '^CNXMEDIA',
    'Pharma'                            : '^CNXPHARMA',
    'Services'                          : '^CNXSERVICE',
    'Construction'                      : '^CNXINFRA',
    'Power'                             : '^CNXENERGY',
    'Defence'                           : 'MID150BEES.NS',
    'Telecommunication'                 : 'MID150BEES.NS',
    'Diversified'                       : 'MID150BEES.NS',
    'Forest Materials'                  : 'MID150BEES.NS',
    'Agriculture'                       : 'MID150BEES.NS',
    'Cement'                            : '^CNXCMDT',
}

print("Fetching sector index/ETF data...")
sector_price   = {}
ticker_data    = {}
unique_tickers = list(set(SECTOR_INDEX_MAP.values()))

for ticker in unique_tickers:
    try:
        df = yf.Ticker(ticker).history(period='1y')
        if len(df) > 50:
            ticker_data[ticker] = df
            print(f"  ✅ {ticker:20} {len(df)} days")
        else:
            print(f"  ⚠  {ticker:20} only {len(df)} days — FAILED")
    except Exception as e:
        print(f"  ❌ {ticker:20} ERROR: {e}")

fallback = 'MID150BEES.NS'
for sector, ticker in SECTOR_INDEX_MAP.items():
    if ticker in ticker_data:
        sector_price[sector] = ticker_data[ticker]
    elif fallback in ticker_data:
        sector_price[sector] = ticker_data[fallback]

print(f"\nUnique tickers fetched : {len(ticker_data)}")
print(f"Sectors mapped         : {len(sector_price)}/{len(SECTOR_INDEX_MAP)}")



Fetching sector index/ETF data...
  ✅ ^CNXREALTY           247 days
  ✅ ^CNXSERVICE          247 days
  ✅ MID150BEES.NS        248 days
  ✅ ^CNXIT               248 days
  ✅ PVTBANIETF.NS        248 days
  ✅ ^CNXFMCG             247 days
  ✅ ^CNXCMDT             247 days
  ✅ ^CNXMEDIA            247 days
  ✅ ^CNXPHARMA           247 days
  ✅ ^CNXINFRA            247 days
  ✅ ^CNXMETAL            247 days
  ✅ ^CNXAUTO             247 days
  ✅ ^NSEBANK             248 days
  ✅ ^CNXPSUBANK          247 days
  ✅ FINIETF.NS           248 days
  ✅ ^CNXENERGY           247 days
  ✅ ^CNXCONSUM           247 days

Unique tickers fetched : 17
Sectors mapped         : 27/27


In [6]:
# ── CELL 6: SECTOR INDICATORS + TREND ────────────────────────
def calculate_sector_indicators(df):
    data = df.copy()
    data['EMA20']  = data['Close'].ewm(span=20,  adjust=False).mean()
    data['EMA50']  = data['Close'].ewm(span=50,  adjust=False).mean()
    data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()

    delta    = data['Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs       = avg_gain / (avg_loss + 1e-9)
    data['RSI'] = 100 - (100 / (1 + rs))

    ema12             = data['Close'].ewm(span=12, adjust=False).mean()
    ema26             = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD']      = ema12 - ema26
    data['Signal']    = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['Signal']

    high     = data['High']
    low      = data['Low']
    close    = data['Close']
    tr       = pd.concat([high - low,
                          (high - close.shift()).abs(),
                          (low  - close.shift()).abs()], axis=1).max(axis=1)
    dm_plus  = high.diff()
    dm_minus = -low.diff()
    dm_plus  = dm_plus.where((dm_plus > dm_minus) & (dm_plus > 0), 0)
    dm_minus = dm_minus.where((dm_minus > dm_plus) & (dm_minus > 0), 0)
    atr      = tr.ewm(span=14, adjust=False).mean()
    di_plus  = 100 * dm_plus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    di_minus = 100 * dm_minus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    dx       = 100 * (di_plus - di_minus).abs() / (di_plus + di_minus + 1e-9)
    data['ADX']      = dx.ewm(span=14, adjust=False).mean()
    data['DI_Plus']  = di_plus
    data['DI_Minus'] = di_minus

    return data


def get_sector_trend(sector):
    if sector not in sector_price:
        return 'No data', '—'

    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]

    close    = latest['Close']
    rsi      = round(latest['RSI'],   1)
    adx      = round(latest['ADX'],   1)
    ema20    = latest['EMA20']
    ema50    = latest['EMA50']
    ema200   = latest['EMA200']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    macd     = latest['MACD']
    signal   = latest['Signal']
    macd_h   = latest['MACD_Hist']

    above_ema200 = close > ema200
    above_ema50  = close > ema50
    ema_aligned  = ema20 > ema50 > ema200
    ema_bearish  = ema20 < ema50 < ema200
    uptrend      = di_plus > di_minus
    macd_bull    = macd > signal and macd_h > 0
    macd_bear    = macd < signal and macd_h < 0
    macd_turning = macd_h > 0 and macd < signal

    bull_score = sum([above_ema200, above_ema50, ema_aligned,
                      uptrend, macd_bull, rsi > 55])
    bear_score = sum([not above_ema200, not above_ema50, ema_bearish,
                      not uptrend, macd_bear, rsi < 45])

    if   bull_score >= 5 and adx > 25: label = 'Strong Uptrend ↑↑'
    elif bull_score >= 4 and adx > 20: label = 'Uptrend ↑'
    elif bull_score >= 3 and adx <= 20: label = 'Weak Uptrend →↑'
    elif bull_score == bear_score or adx < 15: label = 'Sideways →'
    elif bear_score >= 5 and adx > 25: label = 'Strong Downtrend ↓↓'
    elif bear_score >= 4 and adx > 20: label = 'Downtrend ↓'
    elif bear_score >= 3 and adx <= 20: label = 'Weak Downtrend →↓'
    else: label = 'Sideways →'

    if macd_turning and bear_score >= 3:
        label += ' (MACD turning)'

    ema_str  = '↑↑↑' if ema_aligned else '↓↓↓' if ema_bearish else 'mixed'
    macd_str = '▲' if macd_bull else '▼' if macd_bear else '~'
    detail   = (f"RSI {rsi} | ADX {adx} | MACD {macd_str} | "
                f"EMA {ema_str} | {'Above' if above_ema200 else 'Below'} EMA200")

    return label, detail



In [7]:
# ── CELL 7: BEARISH REVERSAL SCORER ──────────────────────────
def score_bearish_reversal(df):
    """
    5 strict filters — signal fires at 3+/5
    F1: Runup > 30% from 52w low
    F2: RSI > 70 for 5 consecutive days
    F3: RSI divergence AND MACD divergence (both required)
    F4: ADX was > 30, now declined 15%+
    F5: Volume declining on up days
    """
    if len(df) < 60:
        return False, 0, {}

    close  = df['Close']
    rsi    = df['RSI']
    macd_h = df['MACD_Hist']
    adx    = df['ADX']
    volume = df['Volume']

    details = {}
    score   = 0

    # Filter 1: Runup > 30% from 52w low
    low_52w   = close.iloc[-252:].min() if len(close) >= 252 else close.min()
    current   = close.iloc[-1]
    runup_pct = (current - low_52w) / low_52w * 100
    f1        = runup_pct >= 30
    details['runup_pct']       = round(runup_pct, 1)
    details['filter1_runup']   = f1
    if f1: score += 1

    # Filter 2: RSI > 70 for 5 consecutive days
    rsi_last5 = rsi.iloc[-5:]
    f2        = (rsi_last5 > 70).all()
    details['rsi_last5_min']         = round(rsi_last5.min(), 1)
    details['filter2_rsi_extended']  = f2
    if f2: score += 1

    # Filter 3: RSI divergence + MACD divergence (BOTH required)
    price_high_recent = close.iloc[-10:].max()
    price_high_prev   = close.iloc[-20:-10].max()
    rsi_high_recent   = rsi.iloc[-10:].max()
    rsi_high_prev     = rsi.iloc[-20:-10].max()
    rsi_div           = (price_high_recent > price_high_prev) and \
                        (rsi_high_recent   < rsi_high_prev)

    macd_high_recent = macd_h.iloc[-10:].max()
    macd_high_prev   = macd_h.iloc[-20:-10].max()
    macd_div         = (price_high_recent > price_high_prev) and \
                       (macd_high_recent  < macd_high_prev)

    f3 = rsi_div and macd_div
    details['rsi_divergence']   = rsi_div
    details['macd_divergence']  = macd_div
    details['filter3_both_div'] = f3
    if f3: score += 1

    # Filter 4: ADX was > 30, now declining 15%+
    adx_peak = adx.iloc[-15:-5].max()
    adx_now  = adx.iloc[-1]
    f4       = (adx_peak > 30) and (adx_now < adx_peak * 0.85)
    details['adx_peak']         = round(adx_peak, 1)
    details['adx_now']          = round(adx_now,  1)
    details['filter4_adx_fade'] = f4
    if f4: score += 1

    # Filter 5: Volume declining on up days
    recent_up_vol = volume.iloc[-10:][close.iloc[-10:].diff() > 0].mean()
    prev_up_vol   = volume.iloc[-20:-10][close.iloc[-20:-10].diff() > 0].mean()
    vol_declining = (recent_up_vol < prev_up_vol * 0.85) if prev_up_vol > 0 else False
    f5            = vol_declining
    details['vol_ratio_up_days']   = round(recent_up_vol / prev_up_vol, 2) \
                                     if prev_up_vol > 0 else 0
    details['filter5_vol_decline'] = f5
    if f5: score += 1

    signal = score >= 3
    return signal, score, details



In [8]:
# ── CELL 8: RUN ALL SIGNALS ───────────────────────────────────
# Volume inference
for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference

# Sector trend
for idx, row in tech_df.iterrows():
    s_label, s_detail = get_sector_trend(row['Sector'])
    tech_df.at[idx, 'Sector Trend']  = s_label
    tech_df.at[idx, 'Sector Detail'] = s_detail

# Bearish reversal
bearish_results = {}
for symbol, df in indicator_data.items():
    if symbol not in tech_df['Symbol'].values:
        continue
    signal, score, details = score_bearish_reversal(df)
    bearish_results[symbol] = {'signal': signal, 'score': score, 'details': details}

triggered = [s for s, v in bearish_results.items() if v['signal']]
print(f"Vol inference  : done ✅")
print(f"Sector trends  : done ✅")
print(f"Bearish signals: {triggered if triggered else 'None this week'} ✅")


Vol inference  : done ✅
Sector trends  : done ✅
Bearish signals: ['BEL', 'ABSLAMC', 'PARAS'] ✅


In [12]:
# ── CELL 9: HELPER FUNCTIONS FOR REPORT ──────────────────────
CAP_ORDER = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

def get_reason(row):
    setup  = str(row['Best Setup'])
    rsi    = float(row['RSI'])
    adx    = float(row['ADX'])
    macd   = float(row['MACD Hist'])
    price  = float(row['Current Price'])
    ema20  = float(row['EMA20'])
    ema50  = float(row['EMA50'])
    ema200 = float(row['EMA200'])

    above_ema200 = price > ema200
    above_ema50  = price > ema50
    ema_aligned  = ema20 > ema50 > ema200  # full bullish alignment

    if setup == 'Momentum':
        if ema_aligned and rsi > 55:
            return f'Strong uptrend, EMA aligned, RSI {rsi:.0f}'
        elif above_ema200 and rsi > 50:
            return f'Momentum building, above EMA200, RSI {rsi:.0f}'
        elif above_ema200 and rsi <= 50:
            return f'Consolidating above EMA200, RSI {rsi:.0f}'
        elif not above_ema200 and rsi > 60:
            return f'Short bounce, below EMA200, RSI {rsi:.0f}'
        else:
            return f'Weak momentum, below EMA200, RSI {rsi:.0f}'

    elif setup == 'Reversal':
        if rsi < 30:   return f'Deeply oversold RSI {rsi:.0f}, reversal imminent'
        elif rsi < 35: return f'Oversold RSI {rsi:.0f}, watch for turn'
        elif macd > 0: return f'RSI {rsi:.0f}, MACD turning up'
        else:          return f'Divergence forming, RSI {rsi:.0f}'

    else:  # Watching
        if adx > 40:   return f'Strong downtrend ADX {adx:.0f}, wait'
        elif adx > 25: return f'Downtrend ADX {adx:.0f}, no setup yet'
        elif rsi < 30: return f'Deeply oversold RSI {rsi:.0f}, watch for turn'
        elif rsi < 40: return f'Oversold RSI {rsi:.0f}, watch for turn'
        else:          return f'RSI {rsi:.0f}, setup not confirmed'


def get_breakout_vol_inference(week_vol, break_vol):
    if   break_vol >= 2.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡⚡ → Strong breakout volume building"
    elif break_vol >= 1.5: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡  → Volume building toward breakout"
    elif break_vol >= 1.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ~  → Normal volume, no breakout pressure"
    elif break_vol >= 0.7: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Quiet, below consolidation avg"
    else:                  return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Very quiet, no breakout interest"


def print_sector_summary():
    print(f"\n{'─'*78}")
    print(f"  SECTOR TREND SUMMARY  (full market)")
    print(f"{'─'*78}")
    print(f"  {'Sector':42} {'Trend':28} Detail")
    print(f"  {'─'*76}")
    for sector in sorted(SECTOR_INDEX_MAP.keys()):
        label, detail = get_sector_trend(sector)
        print(f"  {sector:42} {label:28} {detail}")


def print_bearish_reversal_section():
    print(f"\n{'─'*78}")
    print(f"  ⚠  BEARISH REVERSAL WATCH")
    print(f"  Stocks showing distribution signals — consider partial profit booking (25-30%)")
    print(f"  Signal fires when 3 or more of 5 filters trigger")
    print(f"{'─'*78}")
    print(f"  Filters: (1)Runup>30%  (2)RSI>70×5d  (3)RSI+MACD divergence")
    print(f"           (4)ADX fading  (5)Volume declining on up days")
    print(f"{'─'*78}")

    triggered_stocks = [(s, v) for s, v in bearish_results.items() if v['signal']]

    if not triggered_stocks:
        print(f"\n  No stocks triggered this week. Market is clean.")
    else:
        for symbol, result in triggered_stocks:
            score   = result['score']
            details = result['details']
            row     = tech_df[tech_df['Symbol'] == symbol].iloc[0]
            f1 = '✓' if details.get('filter1_runup')          else '✗'
            f2 = '✓' if details.get('filter2_rsi_extended')   else '✗'
            f3 = '✓' if details.get('filter3_both_div')       else '✗'
            f4 = '✓' if details.get('filter4_adx_fade')       else '✗'
            f5 = '✓' if details.get('filter5_vol_decline')    else '✗'
            if   score == 5: inference = 'Very strong distribution signal → book 25-30% immediately'
            elif score == 4: inference = 'Strong signal → start scaling out 20-25%'
            else:            inference = 'Early warning → monitor closely, avoid adding'
            print(f"""
  ⚠  {symbol}  ({row['Sector']})  Rs{row['Current Price']:.2f}  Score:{score}/5
     Runup from 52w low : {details.get('runup_pct', 0):+.1f}%
     Filters fired      : (1){f1} Runup  (2){f2} RSI>70×5d  (3){f3} Divergence  (4){f4} ADX fade  (5){f5} Vol↓
     RSI divergence     : {details.get('rsi_divergence')}  |  MACD divergence: {details.get('macd_divergence')}
     ADX peak→now       : {details.get('adx_peak', 0):.1f} → {details.get('adx_now', 0):.1f}  (fading trend strength)
     Sector             : {row['Sector Trend']} | {row['Sector Detail']}
     Action             : {inference}""")

    print(f"\n{'─'*78}")
    print(f"  Note: This is NOT a sell signal. Quality stocks may continue higher.")
    print(f"  Use to protect partial gains. Re-enter if pullback holds key levels.")
    print(f"{'─'*78}")

print("✅ Helper functions loaded")

✅ Helper functions loaded


In [15]:
# inserting range percentage to report

# ── Fix: add range_pct from consolidation_info ────────────────
consol_info_df = pd.read_csv('../data/consolidation_info.csv')
print("Consolidation info columns:", consol_info_df.columns.tolist())

# Merge range_pct into tech_df
tech_df = tech_df.merge(
    consol_info_df[['Symbol', 'range_pct']],
    on='Symbol',
    how='left'
)
tech_df['range_pct'] = tech_df['range_pct'].fillna(0.0)

print(f"range_pct added ✅")
print(tech_df[tech_df['In Consolidation'] == True][['Symbol', 'Consol Days', 'range_pct', 'Pct to Breakout']].to_string(index=False))

Consolidation info columns: ['Symbol', 'Fund Score', 'Sector', 'consol_days', 'duration_label', 'range_pct', 'range_high', 'range_low', 'pct_to_breakout', 'is_breaking_out', 'breakout_volume', 'has_bell', 'current_price']
range_pct added ✅
    Symbol  Consol Days  range_pct  Pct to Breakout
PERSISTENT          390      38.43           -26.90
   FINEORG          799      36.97           -21.91
 SUNPHARMA          527      26.17            -4.01
  DIVISLAB          332      23.57           -10.24
 GARFIBRES          544      49.23           -34.26
       HAL          490      49.79           -26.29
  GRANULES          444      47.50           -11.30
  RELIANCE          612      34.98            -9.32
   IPCALAB          486      37.72            -4.28
  SUPRAJIT          497      38.87           -24.40


In [16]:
# ── CELL 10: PRINT WEEKLY REPORT ─────────────────────────────
today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg (breakout signal)")

# ── Sector Summary ────────────────────────────────────────────
print_sector_summary()

# ── Tier 1 ────────────────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1['_cap_order'] = tier1['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier1 = tier1.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

serial = 1
current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short   = {'Large Cap':'L','Mini Large Cap':'ML','Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        bvol_inf    = get_breakout_vol_inference(row['Vol 5D Ratio'], row['Breakout Vol'])
        consol_line = (f"\n    Base    : {row['Consol Days']}d "
                      f"({row['Consol Label']}) | "
                      f"{row['Pct to Breakout']:+.1f}% to breakout"
                      f"\n    Break   : {bvol_inf}")

    print(f"""
  #{serial}  {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr  Rs{row['Current Price']:.2f}
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10{consol_line}
    Tech    : RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}  → {get_reason(row)}
    Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
    serial += 1

# ── Tier 2A ───────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")

for _, row in tier2a.iterrows():
    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : [{row['Best Setup']}|{row['Tech Score']:.0f}]  "
          f"RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
          f"→ {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── Tier 2B ───────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")

for _, row in tier2b.iterrows():
    bvol_inf = get_breakout_vol_inference(row['Vol 5D Ratio'], row['Breakout Vol'])
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"Range:{row['range_pct']:.1f}%  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {bvol_inf}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── Tier 3 ────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3['_cap_order'] = tier3['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier3 = tier3.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short   = {'Large Cap':'L','Mini Large Cap':'ML','Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''

    if row['In Consolidation'] and row['Breakout Vol'] > 0:
        vol_line = get_breakout_vol_inference(row['Vol 5D Ratio'], row['Breakout Vol'])
    else:
        vol_line = (f"Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
                   f"→ {row['Vol Inference']}")

    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {vol_line}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── Long Base Watch ───────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high — Break Vol > 1.5x = signal building")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Days':6} {'Range%':7} {'ToBreak':8} "
      f"{'WkVol':7} {'BrkVol':7}  Signal & Status")
print(f"  {'─'*90}")

for _, row in consol_stocks.iterrows():
    pct      = row['Pct to Breakout']
    status   = 'NEAR BREAKOUT ⚡' if pct > -5 else \
               'Approaching'      if pct > -15 else 'Early stage'
    bvol     = row['Breakout Vol']
    wvol     = row['Vol 5D Ratio']
    bvol_tag = '⚡ BUILDING' if bvol >= 1.5 else \
               '~ normal'    if bvol >= 0.8 else '→ quiet'
    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}%  "
          f"{wvol:.2f}x  "
          f"{bvol:.2f}x  "
          f"{bvol_tag:12}  [{status}]")

# ── Bearish Reversal Watch ────────────────────────────────────
print_bearish_reversal_section()

# ── Footer ────────────────────────────────────────────────────
print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank  = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg")
print(f"{'─'*78}")

# ── Save ──────────────────────────────────────────────────────
tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  22 March 2026

  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap
  Week Vol = this week avg vs prior 15-day avg
  Break Vol= this week avg vs consolidation period avg (breakout signal)

──────────────────────────────────────────────────────────────────────────────
  SECTOR TREND SUMMARY  (full market)
──────────────────────────────────────────────────────────────────────────────
  Sector                                     Trend                        Detail
  ────────────────────────────────────────────────────────────────────────────
  Agriculture                                Strong Downtrend ↓↓          RSI 34.4 | ADX 42.3 | MACD ▼ | EMA mixed | Below EMA200
  Automobile and Auto Components             Strong Downtrend ↓↓          RSI 37.2 | ADX 40.3 | MACD ▼ | EMA mixed | Below EMA200
  Banking                                    Strong Downtrend ↓↓          RSI 26.1 | ADX 53.8 | MACD ▼ | EMA mixed | Below EMA200
  